## Gold daily invoice details snapshot approach

This notebook builds the gold table [erp_lakehouse.gold.dailyinvoicedetails](#table) at invoice-line grain by combining [erp_lakehouse.silver.sl_invoiceheader](#table) and [erp_lakehouse.silver.sl_invoiceline](#table).

### Target design
* Grain: one row per invoice line per snapshot day.
* Snapshot behavior: keep historical daily versions by adding a `snapshot_date` and `snapshot_loaded_at` on every run instead of updating prior rows in place.
* Idempotent daily rerun: for the current run date only, delete the existing snapshot partition and reload it. This preserves earlier snapshot dates while preventing duplicates if the notebook runs multiple times on the same day.
* Join pattern: use invoice line as the driving table and enrich from invoice header on `CompanyKey`, `BrandKey`, and `InvKey`.

### Columns included
* Header dimensions: company, brand, invoice, sales rep, customer, bill-to, ship-to, sell-to, invoice type/status, invoice dates, currency, exchange-rate, and header-level amount context.
* Line dimensions: invoice line, sales order, warehouse, product line, item, UOM, ship date, exchange date, and record status.
* Line metrics: all numeric line measures from the silver line table, including quantity, unit cost, unit price, discounted unit price, line cost, line value, discount, tax, and exchange-rate measures across CC, LC, and IN currencies.
* Derived metrics: profit is derived from line value minus line cost for CC, LC, and IN; margin percent is calculated where line value is non-zero.

### Load logic
1. Create schema `erp_lakehouse.gold` if it does not already exist.
2. Read the current state from silver header and line tables.
3. Build a joined snapshot dataset at invoice-line grain.
4. Add snapshot metadata columns.
5. Create the gold Delta table on first run, partitioned by `snapshot_date`.
6. On later runs, remove only today's snapshot and append the refreshed daily snapshot.

### Notes
* `SourceUpdatedTime` in the line table appears to be null for many rows, so the notebook uses `coalesce(SourceUpdatedTime, SysCreatedTime)` for both source audit fields and the consolidated `source_last_updated_at`.
* The table is intended as a historical daily snapshot table, not a current-state overwrite table.
* Because the grain is invoice line by snapshot day, a single invoice line can appear multiple times across different snapshot dates as its status or amounts evolve.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

spark.sql("CREATE SCHEMA IF NOT EXISTS erp_lakehouse.gold")

decimal_type = DecimalType(15, 2)

snapshot_date_expr = F.current_date()
snapshot_loaded_at_expr = F.current_timestamp()
target_table = "erp_lakehouse.gold.dailyinvoicedetails"

header_df = (
    spark.table("erp_lakehouse.silver.sl_invoiceheader")
    .alias("h")
)

line_df = (
    spark.table("erp_lakehouse.silver.sl_invoiceline")
    .alias("l")
)

joined_df = (
    line_df.join(
        header_df,
        on=["CompanyKey", "BrandKey", "InvKey"],
        how="left"
    )
    .select(
        snapshot_date_expr.alias("snapshot_date"),
        snapshot_loaded_at_expr.alias("snapshot_loaded_at"),
        F.coalesce(F.col("l.SourceUpdatedTime"), F.col("l.SysCreatedTime"), F.col("h.SourceUpdatedTime"), F.col("h.SysCreatedTime")).alias("source_last_updated_at"),
        F.coalesce(F.col("l.RecordStatus"), F.col("h.RecordStatus")).alias("source_record_status"),

        F.col("h.CompanyKey").alias("company_key"),
        F.col("h.BrandKey").alias("brand_key"),
        F.col("h.InvKey").alias("invoice_key"),
        F.col("h.InvSrcId").alias("invoice_source_id"),
        F.col("l.InvLnSrcId").alias("invoice_line_source_id"),
        F.col("l.WHSrcId").alias("warehouse_source_id"),
        F.col("l.ProductLnSrcId").alias("product_line_source_id"),
        F.col("l.ItemSrcId").alias("item_source_id"),
        F.col("l.UOMSrcId").alias("uom_source_id"),
        F.col("h.SalesRepInSrcId").alias("sales_rep_source_id"),
        F.col("h.CustSrcId").alias("customer_source_id"),
        F.col("l.InvLnKey").alias("invoice_line_key"),
        F.col("h.SalesRepInKey").alias("sales_rep_key"),
        F.col("h.CustKey").alias("customer_key"),
        F.col("l.ProductLnKey").alias("product_line_key"),
        F.col("l.ItemKey").alias("item_key"),

        F.col("h.InvType").alias("invoice_type"),
        F.col("h.InvTypeDesc").alias("invoice_type_desc"),
        F.col("h.InvStatus").alias("invoice_status"),
        F.col("h.InvPaidStatus").alias("invoice_paid_status"),
        F.col("l.InvLnStatus").alias("invoice_line_status"),
        F.col("l.InvLnType").alias("invoice_line_type"),
        F.col("l.InvLnTypeDesc").alias("invoice_line_type_desc"),
        F.col("h.InvDt").alias("invoice_date"),
        F.col("h.InvPostedDt").alias("invoice_posted_date"),
        F.col("h.InvDueDt").alias("invoice_due_date"),
        F.col("l.ShipDt").alias("ship_date"),
        F.col("h.ExDt").alias("exchange_date"),
        F.col("h.CustName").alias("customer_name"),
        F.col("h.CustContactName").alias("customer_contact_name"),
        F.col("h.CustBillToCity").alias("customer_bill_to_city"),
        F.col("h.CustBillToState").alias("customer_bill_to_state"),
        F.col("h.CustBillToCountry").alias("customer_bill_to_country"),
        F.col("h.CustShipToCity").alias("customer_ship_to_city"),
        F.col("h.CustShipToState").alias("customer_ship_to_state"),
        F.col("h.CustShipToCountry").alias("customer_ship_to_country"),
        F.col("h.CustSellToCity").alias("customer_sell_to_city"),
        F.col("h.CustSellToState").alias("customer_sell_to_state"),
        F.col("h.CustSellToCountry").alias("customer_sell_to_country"),
        F.col("l.ProductLnDesc").alias("product_line_desc"),
        F.col("l.ProductLnType").alias("product_line_type"),
        F.col("l.ItemDesc").alias("item_desc"),
        F.col("l.ItemType").alias("item_type"),
        F.col("l.UOM").alias("uom"),
        F.col("l.CustCurr").alias("customer_currency"),
        F.col("l.LocCurr").alias("local_currency"),
        F.col("l.ExRtLCCC").cast(decimal_type).alias("exchange_rate_lc_cc"),
        F.col("l.ExRtCCLC").cast(decimal_type).alias("exchange_rate_cc_l
        F.col("l.UnitCostLC").alias("unit_cost_lc"),
        F.col("l.UnitCostIN").alias("unit_cost_in"),
        F.col("l.UnitPriceCC").alias("unit_price_cc"),
        F.col("l.UnitPriceLC").alias("unit_price_lc"),
        F.col("l.UnitPriceIN").alias("unit_price_in"),
        F.col("l.DiscUnitPriceCC").alias("discounted_unit_price_cc"),
        F.col("l.DiscUnitPriceLC").alias("discounted_unit_price_lc"),
        F.col("l.DiscUnitPriceIN").alias("discounted_unit_price_in"),
        F.col("l.InvLnCostCC").alias("line_cost_cc"),
        F.col("l.InvLnCostLC").alias("line_cost_lc"),
        F.col("l.InvLnCostIN").alias("line_cost_in"),
        F.col("l.InvLnValCC").alias("line_value_cc"),
        F.col("l.InvLnValLC").alias("line_value_lc"),
        F.col("l.InvLnValIN").alias("line_value_in"),
        F.col("l.InvLnDiscPerc").alias("line_discount_percent"),
        F.col("l.InvLnDiscValCC").alias("line_discount_value_cc"),
        F.col("l.InvLnDiscValLC").alias("line_discount_value_lc"),
        F.col("l.InvLnDiscValIN").alias("line_discount_value_in"),
        F.col("l.InvLnTaxPerc").alias("line_tax_percent"),
        F.col("l.InvLnTaxValCC").alias("line_tax_value_cc"),
        F.col("l.InvLnTaxValLC").alias("line_tax_value_lc"),
        F.col("l.InvLnTaxValIN").alias("line_tax_value_in"),
        (F.col("l.InvLnValCC") - F.col("l.InvLnCostCC")).alias("line_profit_cc"),
        (F.col("l.InvLnValLC") - F.col("l.InvLnCostLC")).alias("line_profit_lc"),
        (F.col("l.InvLnValIN") - F.col("l.InvLnCostIN")).alias("line_profit_in"),
        F.col("h.SourceUpdatedTime").alias("header_source_updated_time"),
        F.col("h.SysCreatedTime").alias("header_sys_created_time"),
        F.col("h.RecordStatus").alias("header_record_status"),
        F.col("l.SourceUpdatedTime").alias("line_source_updated_time"),
        F.col("l.SysCreatedTime").alias("line_sys_created_time"),
        F.col("l.RecordStatus").alias("line_record_status")
    )
)

In [0]:
table_location = "abfss://gold@bbmanufacturingprod.dfs.core.windows.net/delta/dailyinvoicedetails"

current_snapshot_date = spark.sql("SELECT current_date() AS snapshot_date").collect()[0]["snapshot_date"]

if spark.catalog.tableExists(target_table):
    spark.sql(f"DELETE FROM {target_table} WHERE snapshot_date = DATE('{current_snapshot_date}')")
    joined_df.write.format("delta").mode("append").option("mergeSchema", "true").option("path", table_location).saveAsTable(target_table)
else:
    (
        joined_df.write.format("delta")
        .mode("overwrite")
        .partitionBy("snapshot_date")
        .option("path", table_location)
        .saveAsTable(target_table)
    )

print(f"Loaded snapshot for {current_snapshot_date} into {target_table}")
print(f"Rows loaded: {joined_df.count()}")


In [0]:
display(
    joined_df.select(
        "snapshot_date",
        "invoice_date",
        "invoice_key",
        "invoice_line_key",
        "customer_name",
        "item_desc",
        "line_qty",
        "line_value_lc",
        "line_cost_lc",
        "line_profit_lc"
    ).orderBy(F.col("source_last_updated_at").desc_nulls_last()).limit(20)
)